# Translation of data - NL2EN

In [11]:
%pip install transformers accelerate
%pip install ctranslate2 OpenNMT-py==2.* sentencepiece
%pip install sacremoses

Note: you may need to restart the kernel to use updated packages.
zsh:1: no matches found: OpenNMT-py==2.*
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [1]:
#%pip install transformers -U
import transformers

In [1]:
import ctranslate2

#https://opennmt.net/CTranslate2/guides/transformers.html#marianmt
!ct2-transformers-converter --model Helsinki-NLP/opus-mt-nl-en --output_dir /workspace/mijnidbcoachnlp/data/ct2_model --force

model.safetensors: 100%|██████████████████████| 316M/316M [00:01<00:00, 286MB/s]


In [1]:
# Load dataset
import pandas as pd
import tqdm

path = "/workspace/persistent/mijnidbcoachnlp/data/analysis_data/sentence_data_for_analysis.xlsx"
df = pd.read_excel(path, index_col=0)
df.tail()

,message_id,sentence,clean_sentence,sentence_id
41114,32686,Volgens mij is dat in februari geweest.,Volgens mij is dat in februari geweest.,41115
41115,32686,[LOCATIE] een lange tijd terug.,een lange tijd terug.,41116
41116,32686,Om de hoeveel tijd vinden jullie dat ik bloed ...,Om de hoeveel tijd vinden jullie dat ik bloed ...,41117
41117,32687,"Hallo [PERSOON] Gisteren, laat in de middag, p...","Gisteren, laat in de middag, potje en formulie...",41118
41118,32687,Heb nog veel en waterige diaree Ben dus benieu...,Heb nog veel en waterige diaree Ben dus benieu...,41119


In [3]:
messages = df["sentence"].tolist()
len(messages)
#sentences = df["sentence"].tolist()

41119

In [2]:
# drop na's of clean_message because they will not be processed in the translations
df = df.dropna(subset=["sentence"])
len(df)

41119

In [ ]:
import ctranslate2
import transformers
import tqdm

# Load CTranslate2 Translator and the tokenizer
translator = ctranslate2.Translator('/workspace/persistent/mijnidbcoachnlp/data/ct2_model')
tokenizer = transformers.AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-nl-en")

# Function to translate a batch of messages using CTranslate2
def translate_batch(messages, batch_size=1000):
    translations = []
    
    for i in tqdm.tqdm(range(0, len(messages), batch_size)):
        batch = messages[i:i + batch_size]
        
        # Tokenize the batch of messages into tokens (lists of token strings)
        source_batch = [tokenizer.convert_ids_to_tokens(tokenizer.encode(message, add_special_tokens=True)) for message in batch]

        # Translate the batch using CTranslate2 (expects a list of tokenized inputs)
        results = translator.translate_batch(source_batch)

        # Decode the hypotheses (translations) back to text
        for result in results:
            translated_tokens = result.hypotheses[0]  # Access the first hypothesis (most likely translation)
            translated_text = tokenizer.convert_tokens_to_string(translated_tokens)  # Convert tokens back to string
            translations.append(translated_text)
        
        # Print original and translated messages
        #for original, translated in zip(batch, translations[-len(batch):]):
            #print(f"Original: {original}")
            #print(f"Translated: {translated}")
    
    return translations

# Function to clean and filter valid text messages
def clean_messages(messages):
    return [str(message) for message in messages if isinstance(message, str) and message.strip()]

# Clean the messages to remove any non-string or empty values
cleaned_messages = clean_messages(messages)

# Translate messages in batches using CTranslate2
translated_messages = translate_batch(cleaned_messages)

# Add the translations back into the DataFrame (optional)
df["translated_message"] = [None if not isinstance(msg, str) else translated_messages.pop(0) for msg in messages]

# Save the translated messages to a new file (optional)
df.to_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/clean_message_data.xlsx", index=False)


In [10]:
import ctranslate2
import transformers
import tqdm
import pandas as pd
import os

# Settings
BATCH_SIZE = 100
SAVE_EVERY = 1  # Save after every batch
TRANSLATED_FILE = "/workspace/persistent/mijnidbcoachnlp/data/analysis_data/partial_sentence_translations.csv"
MAX_LEN = 512
SKIPPED_LOG = "/workspace/persistent/mijnidbcoachnlp/data/analysis_data/skipped_sentences_due_to_length.txt"

# Load model & tokenizer
translator = ctranslate2.Translator('/workspace/persistent/mijnidbcoachnlp/data/ct2_model')
tokenizer = transformers.AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-nl-en")

# Function to clean and filter valid text messages
def clean_messages(messages):
    return [str(message) for message in messages if isinstance(message, str) and message.strip()]

def translate_batch(batch, global_start_index):
    source_batch = []
    valid_indices = []  # Track indices within the batch for valid messages
    skipped_indices = []

    for local_idx, message in enumerate(batch):
        tokens = tokenizer.encode(message, add_special_tokens=True)
        if len(tokens) <= MAX_LEN:
            token_strs = tokenizer.convert_ids_to_tokens(tokens)
            source_batch.append(token_strs)
            valid_indices.append(local_idx)
        else:
            skipped_indices.append(global_start_index + local_idx)

    results = translator.translate_batch(source_batch)

    # Initialize result list with empty strings
    translations = [""] * len(batch)
    for idx_in_batch, result in zip(valid_indices, results):
        translated_tokens = result.hypotheses[0]
        translated_text = tokenizer.convert_tokens_to_string(translated_tokens)
        translations[idx_in_batch] = translated_text

    # Log skipped messages (append)
    if skipped_indices:
        with open(SKIPPED_LOG, "a") as log_file:
            for idx in skipped_indices:
                log_file.write(f"{idx}\n")

    return translations

cleaned_messages = clean_messages(messages)

# Check if there's a partial file already
if os.path.exists(TRANSLATED_FILE):
    df_partial = pd.read_csv(TRANSLATED_FILE)
    start_idx = len(df_partial)
    print(f"Resuming from sentence {start_idx}")
    translated_messages = df_partial["translated_sentence"].tolist()
else:
    start_idx = 0
    translated_messages = []

# Translate in batches
for i in tqdm.tqdm(range(start_idx, len(messages), BATCH_SIZE)):
     # instead of print(i)
    batch = cleaned_messages[i:i + BATCH_SIZE]
    try:
        translated_batch = translate_batch(batch, i)
        translated_messages.extend(translated_batch)
    except Exception as e:
        print(f"Error translating batch starting at index {i}: {str(e)}")
        translated_messages.extend([""] * len(batch))  # Add empty strings for failed translations

    # Save intermediate results
    if (i // BATCH_SIZE) % SAVE_EVERY == 0:
        df_temp = pd.DataFrame({
            "original_sentence": cleaned_messages[:len(translated_messages)],
            "translated_sentence": translated_messages
        })
        df_temp.to_csv(TRANSLATED_FILE, index=False)

df["translated_sentence"] = translated_messages
# Save final results
#df.to_excel(FINAL_OUTPUT_FILE, index=False)
#print("Translation completed and saved successfully!")

Resuming from sentence 1600


100%|██████████| 396/396 [52:22<00:00,  7.94s/it]


In [12]:
df.tail()

,message_id,sentence,clean_sentence,sentence_id,translated_sentence
41114,32686,Volgens mij is dat in februari geweest.,Volgens mij is dat in februari geweest.,41115,I think that was in February.
41115,32686,[LOCATIE] een lange tijd terug.,een lange tijd terug.,41116,[LOCATION] A long time ago.
41116,32686,Om de hoeveel tijd vinden jullie dat ik bloed ...,Om de hoeveel tijd vinden jullie dat ik bloed ...,41117,"Every once in a while, you think I should have..."
41117,32687,"Hallo [PERSOON] Gisteren, laat in de middag, p...","Gisteren, laat in de middag, potje en formulie...",41118,"Hello [PRESS] Yesterday, late in the afternoon..."
41118,32687,Heb nog veel en waterige diaree Ben dus benieu...,Heb nog veel en waterige diaree Ben dus benieu...,41119,"So Ben has a lot of watery diarrhea left, so I..."


In [14]:
FINAL_OUTPUT_FILE = "/workspace/persistent/mijnidbcoachnlp/data/analysis_data/sentence_data_for_analysis.xlsx"
df.to_excel(FINAL_OUTPUT_FILE, index=True)

### Translate the messages that were too long and skipped 

In [ ]:
import pandas as pd
import numpy as np

# Step 1: Read skipped messages
with open("skipped_due_to_length.txt", "r", encoding="utf-8") as f:
    skipped_messages = [line.strip() for line in f if line.strip()]

# Step 2: Find indices in the DataFrame where 'translated_message' is NaN
na_indices = df[df["translated_message"].isna()].index

# Step 3: Define a translation function (replace with your translator)
def mock_translate(text):
    # Replace with actual translation logic (API call, model, etc.)
    return "[EN] " + text

# Optional: Break messages if too long (depends on your translator's limit)
def split_message(message, max_chars=500):
    # Simple split on sentence boundaries or words
    import textwrap
    return textwrap.wrap(message, max_chars)

# Step 4: Translate, merge, and replace in df
translated_messages = []
for msg in skipped_messages:
    chunks = split_message(msg)
    translated_chunks = [mock_translate(chunk) for chunk in chunks]
    full_translation = " ".join(translated_chunks)
    translated_messages.append(full_translation)

# Step 5: Update the DataFrame
for idx, translation in zip(na_indices, translated_messages):
    df.at[idx, "translated_message"] = translation

# Optional: Save the updated DataFrame
# df.to_csv("df_with_fixed_translations.csv", index=False)
